## Microsoft Agent Framework (MAF): Ramp-Up Training Materials

To get the latest version of MAF Python package, use:

``` bash
pip install --upgrade agent-framework
pip install --upgrade azure-ai-projects
```

This notebook was tested with a specific version of Agent Framework, v1.11.0. To ensure reproducible output, please install Agent Framework with:

``` bash
pip install --force-reinstall agent-framework==1.11.0
```

## 📒 Notebook 1: MCP Hosted Tool

### 🪜 Step 1: Configure environment

In [1]:
# Import required packages
import os
import asyncio
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [2]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Define MCP tool

In [3]:
# Initialise Foundry client
client = FoundryChatClient(
    project_endpoint = PROJECT_ENDPOINT,
    model = MODEL_DEPLOYMENT,
    credential = DefaultAzureCredential(),
)

# Define MCP tool
mcp_doc_tool = client.get_mcp_tool(
    name = "microsoft-learn-mcp-tool",
    url = "https://learn.microsoft.com/api/mcp",
    approval_mode = "never_require"
)

print(f"Created MCP tool: {mcp_doc_tool.server_label}")

Created MCP tool: microsoft-learn-mcp-tool


### 🪜 Step 3: Create AI agent

In [4]:
# Create agent with MCP tool
agent = client.as_agent(
    name = "microsoft-documentation-agent",
    instructions = """
    You are an expert on Microsoft technologies.
    Use the MCP documentation tool to fetch accurate, up-to-date information 
    from Microsoft Learn when answering questions.
    Keep responses concise (max 2-3 paragraphs).
    """,
    tools = [mcp_doc_tool]
)

print(f"Created MS Doc agent: {agent.name}, equipped with MCP tool.")

Created MS Doc agent: microsoft-documentation-agent, equipped with MCP tool.


### 🪜 Step 4: Run the agent

In [5]:
# Test chat agent with a prompt
prompt = "What is the difference between Prompt and Hosted Agents in Foundry?"
print(f"User:\n{prompt}\n")

print(f"Agent:")
async for update in agent.run(prompt, stream=True):
    if update.text:
        print(update.text, end="", flush=True)

User:
What is the difference between Prompt and Hosted Agents in Foundry?

Agent:
The main difference between Prompt agents and Hosted agents in Microsoft Foundry is how they are authored, managed, and run:

- **Prompt agents** are fully managed by Foundry with no code to maintain. They are defined declaratively through the Foundry portal or programmatically via SDK or REST with configuration including instructions, model selection, and tools. Foundry handles the runtime, scaling, and compute automatically. Prompt agents are best for fast starts, internal tools, and production scenarios without the need for custom orchestration or complex logic.

- **Hosted agents** are code-based agents that you build using frameworks like Agent Framework, LangGraph, or your own code. You package your agent as a container image or zip, and Foundry runs it with a managed endpoint, automatic scaling, dedicated identity, session state persistence, and observability. Hosted agents offer control over compu